# Exploration du Dataset AMI pour la Diarisation des Locuteurs

Ce notebook va nous permettre de :
1. Télécharger un échantillon du dataset AMI depuis Hugging Face
2. Afficher l'audio d'un exemple
3. Tracer un graphique montrant qui parle et quand (segments de diarisation)

## Installation des bibliothèques nécessaires

Commençons par installer les bibliothèques dont nous aurons besoin.

In [1]:
# Installation des bibliothèques nécessaires
!pip install datasets librosa matplotlib ipython numpy soundfile plotly

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/9.8 MB ? eta -:--:--  Downloading plotly-6.3.0-py3-none-any.whl (9.8 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 13.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 13.2 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/404.4 KB ? eta -:--:--  Downloading narwhals-2.3.0-py3-none-any.whl (404 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.4/404.4 KB 11.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.4/404.4 KB 11.5 MB/s eta 0:00:00a 0:00:01


## Chargement du Dataset AMI depuis Hugging Face

Nous allons utiliser la bibliothèque `datasets` pour charger le dataset AMI avec la configuration "ihm" (Individual Headset Microphones).

In [ ]:
# Chargement du dataset
from datasets import load_dataset

# Charger le dataset AMI avec la configuration IHM (Individual Headset Microphones)
# et extraire seulement les 3 premiers exemples consécutifs sans les mélanger
ds_full = load_dataset("diarizers-community/ami", "ihm")

# Création d'un nouveau dataset avec seulement 3 exemples pour chaque split
ds = {}
for split in ds_full.keys():
    # Prendre seulement les 3 premiers exemples (consécutifs, sans mélange)
    ds[split] = ds_full[split].select(range(min(3, len(ds_full[split]))))

# Afficher les informations du dataset
print("Dataset original:")
print(ds_full)
print("\nDataset réduit à 3 exemples par split:")
print(ds)

/home/developer/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"



## Exploration de la structure des données

Explorons la structure d'un exemple du dataset pour comprendre les informations disponibles.

In [ ]:
# Examinons les échantillons sélectionnés
def show_samples_overview(ds, split="train"):
    print(f"Aperçu des {len(ds[split])} échantillons du split '{split}':")
    
    for i, example in enumerate(ds[split]):
        if "meeting_id" in example:
            meeting_id = example["meeting_id"]
        elif "id" in example:
            meeting_id = example["id"]
        else:
            meeting_id = f"Exemple {i+1}"
            
        audio_duration = len(example["audio"]["array"]) / example["audio"]["sampling_rate"]
        
        print(f"  - Exemple {i+1}: ID={meeting_id}, Durée={audio_duration:.2f} secondes")

# Examinons la structure d'un exemple de train
train_example = ds["train"][0]
print("Clés disponibles dans un exemple:")
print(train_example.keys())

# Affichons quelques informations sur cet exemple
print("\nInformations sur l'exemple:")
for key in train_example.keys():
    if key != "audio":
        print(f"{key}: {train_example[key]}")

# Pour l'audio, affichons juste les métadonnées
print("\nMétadonnées audio:")
print(f"Format audio: {train_example['audio']['array'].dtype}")
print(f"Taux d'échantillonnage: {train_example['audio']['sampling_rate']} Hz")
print(f"Durée: {len(train_example['audio']['array']) / train_example['audio']['sampling_rate']:.2f} secondes")

# Afficher un aperçu des échantillons sélectionnés dans chaque split
for split in ds.keys():
    show_samples_overview(ds, split)

## Visualisation de l'audio

Maintenant, visualisons le signal audio d'un exemple.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import librosa
import librosa.display

# Prendre un exemple plus court pour une meilleure visualisation (si possible)
short_example = ds["train"][0]  # Nous utiliserons le premier exemple pour simplifier
audio_array = short_example["audio"]["array"]
sampling_rate = short_example["audio"]["sampling_rate"]

# Si l'audio est trop long, prenons seulement les 10 premières secondes
max_duration = 10  # en secondes
if len(audio_array) > max_duration * sampling_rate:
    audio_array = audio_array[:int(max_duration * sampling_rate)]

# Visualiser la forme d'onde
plt.figure(figsize=(14, 5))
librosa.display.waveshow(audio_array, sr=sampling_rate)
plt.title("Forme d'onde audio")
plt.xlabel("Temps (s)")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()

## Visualisation des segments de diarisation (qui parle quand)

Maintenant, traçons un graphique montrant qui parle et quand en utilisant les annotations de diarisation disponibles dans le dataset.

In [ ]:
import plotly.graph_objects as go
from IPython.display import Audio

# Fonction pour lire et tracer les segments de diarisation
def plot_diarization(example):
    audio_array = example["audio"]["array"]
    sampling_rate = example["audio"]["sampling_rate"]
    
    # Obtenir les segments de diarisation
    segments = []
    speakers = []
    
    if "segments" in example:
        for segment in example["segments"]:
            segments.append((segment["start"], segment["end"]))
            speakers.append(segment["speaker"])
    else:
        # Si le format est différent, on adaptera ici
        print("Structure des segments non trouvée. Vérifier les clés disponibles:", example.keys())
        if "annotation" in example:
            print("Structure d'annotation:", example["annotation"].keys())
        return
    
    # Créer un graphique avec Plotly
    fig = go.Figure()
    
    # Ajouter les segments par locuteur avec des couleurs différentes
    unique_speakers = list(set(speakers))
    colors = plt.cm.tab10.colors[:len(unique_speakers)]
    
    for i, speaker in enumerate(unique_speakers):
        speaker_segments = [(start, end) for (start, end), spk in zip(segments, speakers) if spk == speaker]
        
        # Pour chaque segment de ce locuteur
        for start, end in speaker_segments:
            fig.add_trace(go.Scatter(
                x=[start, start, end, end],
                y=[i, i+0.8, i+0.8, i],
                fill="toself",
                mode="lines",
                name=f"Speaker {speaker}",
                line=dict(width=0),
                fillcolor=f"rgba({int(colors[i][0]*255)}, {int(colors[i][1]*255)}, {int(colors[i][2]*255)}, 0.7)"
            ))
    
    # Configurer l'apparence du graphique
    fig.update_layout(
        title="Segments de diarisation - Qui parle quand",
        xaxis_title="Temps (secondes)",
        yaxis_title="Locuteurs",
        yaxis=dict(
            tickmode="array",
            tickvals=list(range(len(unique_speakers))),
            ticktext=[f"Locuteur {s}" for s in unique_speakers]
        ),
        legend_title="Locuteurs",
        height=400
    )
    
    # Afficher le graphique
    fig.show()
    
    # Afficher l'audio pour écoute
    display(Audio(audio_array, rate=sampling_rate))
    
    return fig

# Examiner la structure exacte pour comprendre comment accéder aux segments
example = ds["train"][0]
print("Structure de l'exemple:")
print(example.keys())

if "segments" in example:
    print("\nStructure d'un segment:")
    print(example["segments"][0])
elif "annotation" in example:
    print("\nStructure des annotations:")
    print(example["annotation"].keys())

# Essayons de tracer la diarisation
plot_diarization(example)

## Fonction alternative si la structure du dataset est différente

Si la structure du dataset est différente de ce que nous avons anticipé, voici une fonction plus générique.

In [ ]:
# Fonction alternative pour tracer la diarisation selon la structure réelle du dataset
def plot_diarization_alternative(example):
    audio_array = example["audio"]["array"]
    sampling_rate = example["audio"]["sampling_rate"]
    
    # Adaptation selon la structure réelle du dataset
    if "annotation" in example and "segments" in example["annotation"]:
        segments = []
        speakers = []
        
        for segment in example["annotation"]["segments"]:
            if "start" in segment and "end" in segment and "speaker_id" in segment:
                segments.append((segment["start"], segment["end"]))
                speakers.append(segment["speaker_id"])
    else:
        print("Structure non reconnue. Veuillez vérifier le format du dataset.")
        return
    
    # Créer un graphique avec Plotly
    fig = go.Figure()
    
    # Ajouter les segments par locuteur avec des couleurs différentes
    unique_speakers = list(set(speakers))
    colors = plt.cm.tab10.colors[:len(unique_speakers)]
    
    for i, speaker in enumerate(unique_speakers):
        speaker_segments = [(start, end) for (start, end), spk in zip(segments, speakers) if spk == speaker]
        
        # Pour chaque segment de ce locuteur
        for start, end in speaker_segments:
            fig.add_trace(go.Scatter(
                x=[start, start, end, end],
                y=[i, i+0.8, i+0.8, i],
                fill="toself",
                mode="lines",
                name=f"Speaker {speaker}",
                line=dict(width=0),
                fillcolor=f"rgba({int(colors[i][0]*255)}, {int(colors[i][1]*255)}, {int(colors[i][2]*255)}, 0.7)"
            ))
    
    # Configurer l'apparence du graphique
    fig.update_layout(
        title="Segments de diarisation - Qui parle quand",
        xaxis_title="Temps (secondes)",
        yaxis_title="Locuteurs",
        yaxis=dict(
            tickmode="array",
            tickvals=list(range(len(unique_speakers))),
            ticktext=[f"Locuteur {s}" for s in unique_speakers]
        ),
        legend_title="Locuteurs",
        height=400
    )
    
    # Afficher le graphique
    fig.show()
    
    # Afficher l'audio pour écoute
    display(Audio(audio_array, rate=sampling_rate))
    
    return fig

# Essayons cette approche alternative
example = ds["train"][0]
print("Essai avec l'approche alternative:")
plot_diarization_alternative(example)

## Visualisation comparative des 3 échantillons consécutifs

Visualisons maintenant les trois échantillons consécutifs pour comparer leurs caractéristiques audio et de diarisation.

In [ ]:
# Visualisation des 3 échantillons consécutifs du split train
import matplotlib.pyplot as plt
from IPython.display import display, Audio
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def visualize_consecutive_samples(ds, split="train", max_samples=3):
    num_samples = min(max_samples, len(ds[split]))
    
    print(f"Visualisation de {num_samples} échantillons consécutifs du split '{split}'")
    
    for i in range(num_samples):
        example = ds[split][i]
        
        # Afficher les informations de l'échantillon
        if "meeting_id" in example:
            sample_id = example["meeting_id"]
        elif "id" in example:
            sample_id = example["id"]
        else:
            sample_id = f"Exemple {i+1}"
        
        print(f"\n--- Échantillon {i+1} (ID: {sample_id}) ---")
        
        # Extraire l'audio
        audio_array = example["audio"]["array"]
        sampling_rate = example["audio"]["sampling_rate"]
        audio_duration = len(audio_array) / sampling_rate
        
        print(f"Durée audio: {audio_duration:.2f} secondes")
        
        # Si l'audio est trop long, ne prenons que les 10 premières secondes pour la visualisation
        max_display_duration = 10  # en secondes
        if audio_duration > max_display_duration:
            display_array = audio_array[:int(max_display_duration * sampling_rate)]
            print(f"(Affichage limité aux {max_display_duration} premières secondes)")
        else:
            display_array = audio_array
        
        # Visualiser la forme d'onde
        plt.figure(figsize=(14, 3))
        librosa.display.waveshow(display_array, sr=sampling_rate)
        plt.title(f"Forme d'onde audio - Échantillon {i+1}")
        plt.xlabel("Temps (s)")
        plt.ylabel("Amplitude")
        plt.tight_layout()
        plt.show()
        
        # Afficher l'audio pour écoute
        display(Audio(audio_array, rate=sampling_rate))
        
        # Tracer les segments de diarisation si disponibles
        plot_diarization(example)
        
        print("-" * 50)

# Exécuter la visualisation pour les échantillons consécutifs
visualize_consecutive_samples(ds, "train", max_samples=3)

## Conclusion

Dans ce notebook, nous avons :
1. Téléchargé le dataset AMI depuis Hugging Face et limité à 3 échantillons consécutifs par split
2. Exploré la structure des données
3. Visualisé les formes d'onde audio des échantillons
4. Tracé des graphiques montrant qui parle et quand (segments de diarisation)
5. Visualisé les 3 échantillons consécutifs côte à côte pour comparaison

Cette approche nous permet de voir comment les données de diarisation sont structurées dans le dataset et comment les segments de parole sont distribués entre différents locuteurs dans des échantillons consécutifs, ce qui est utile pour comprendre les caractéristiques du dataset et développer des modèles de diarisation de locuteurs.